# Paper 1 (FINAL, Best Model): ConvNeXtLarge Multi-Modal Fusion + Fine-Tuning + Class Weighting + TTA

**Goal:** Maximum realistic accuracy + clear novelty, for IEEE submission.

**Why ConvNeXtLarge:** The base paper's best model was ConvNeXtXLarge (87.62%), which is too large for free Colab. ConvNeXtLarge is the next tier down in the same family -- stronger than ConvNeXtBase, and the closest practical match to the base paper's best architecture.

**Novelty (for your paper):**
1. Multi-modal fusion -- combines image features with patient metadata (age, sex, lesion location), unlike the base paper which uses only images.
2. Fine-tuning -- the base paper used only frozen weights; this model fine-tunes the backbone.
3. Class-weighted loss -- directly addresses the severe class imbalance the base paper did not handle.
4. Test-Time Augmentation (TTA) -- further improves robustness at inference time.

**IMPORTANT: This notebook mounts Google Drive FIRST, so all progress is saved permanently even if the session disconnects.**

**Expected time: 2-2.5 hours** (ConvNeXtLarge is bigger than Base, so each phase takes longer).

## STEP 0: Mount Google Drive (Do This FIRST)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/Paper1_HAM10000'
os.makedirs(SAVE_DIR, exist_ok=True)
print("Drive mounted. All progress will save to:", SAVE_DIR)

## STEP 1: Kaggle Auth + Download Dataset

In [ ]:
!mkdir -p ~/.kaggle
!echo "$KAGGLE_TOKEN" > ~/.kaggle/access_token
!chmod 600 ~/.kaggle/access_token

!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000
!mkdir -p /content/ham10000
!unzip -q skin-cancer-mnist-ham10000.zip -d /content/ham10000
print("Dataset ready!")

## STEP 2: Imports

In [ ]:
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.applications import ConvNeXtLarge
from tensorflow.keras.applications.convnext import preprocess_input as convnext_preprocess
from tensorflow.keras import layers, models, callbacks
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import OneHotEncoder, StandardScaler, label_binarize
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_score, recall_score, accuracy_score, roc_auc_score

print("GPU Available:", tf.config.list_physical_devices('GPU'))

## STEP 3: Load Metadata + Encode (Age, Sex, Localization)

In [ ]:
metadata = pd.read_csv('/content/ham10000/HAM10000_metadata.csv')
all_image_paths = glob.glob('/content/ham10000/**/*.jpg', recursive=True)
image_path_dict = {os.path.splitext(os.path.basename(p))[0]: p for p in all_image_paths}
metadata['image_path'] = metadata['image_id'].map(image_path_dict.get)
metadata = metadata.dropna(subset=['image_path']).reset_index(drop=True)

metadata['age'] = metadata['age'].fillna(metadata['age'].median())
metadata['sex'] = metadata['sex'].fillna('unknown')
metadata['localization'] = metadata['localization'].fillna('unknown')

classes = sorted(metadata['dx'].unique())
class_to_idx = {c: i for i, c in enumerate(classes)}
metadata['label_idx'] = metadata['dx'].map(class_to_idx)

# Encode metadata: age (numeric) + sex (one-hot) + localization (one-hot)
age_scaler = StandardScaler()
metadata['age_scaled'] = age_scaler.fit_transform(metadata[['age']])
sex_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
sex_encoded = sex_encoder.fit_transform(metadata[['sex']])
loc_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
loc_encoded = loc_encoder.fit_transform(metadata[['localization']])

metadata_features = np.concatenate([metadata[['age_scaled']].values, sex_encoded, loc_encoded], axis=1)
META_DIM = metadata_features.shape[1]
for i in range(META_DIM):
    metadata[f'meta_{i}'] = metadata_features[:, i]
meta_cols = [f'meta_{i}' for i in range(META_DIM)]

print("Metadata dimensions (age + sex + localization):", META_DIM)
print("Dataset ready:", metadata.shape)

## STEP 4: Train/Val/Test Split + Class Weights

In [ ]:
train_df, temp_df = train_test_split(metadata, test_size=0.3, stratify=metadata['dx'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['dx'], random_state=42)
print("Train:", train_df.shape, " Val:", val_df.shape, " Test:", test_df.shape)

class_weights_array = compute_class_weight('balanced', classes=np.array(range(len(classes))), y=train_df['label_idx'].values)
class_weights = {i: w for i, w in enumerate(class_weights_array)}
NUM_CLASSES = len(classes)

# Save splits to Drive immediately, in case of disconnect later
test_df.to_csv(f'{SAVE_DIR}/test_df.csv', index=False)
train_df.to_csv(f'{SAVE_DIR}/train_df.csv', index=False)
val_df.to_csv(f'{SAVE_DIR}/val_df.csv', index=False)
print("Splits saved to Drive.")

## STEP 5: Data Pipeline (Image + Metadata Together)

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 8  # small batch size since ConvNeXtLarge is a big model

def make_tf_dataset(df, augment=False, shuffle=False):
    paths = df['image_path'].values
    meta = df[meta_cols].values.astype(np.float32)
    labels = tf.keras.utils.to_categorical(df['label_idx'].values, NUM_CLASSES)

    def load_img(path):
        img = tf.io.read_file(path)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.resize(img, IMG_SIZE)
        if augment:
            img = tf.image.random_flip_left_right(img)
            img = tf.image.random_flip_up_down(img)
            img = tf.image.random_brightness(img, 0.2)
        img = convnext_preprocess(img)
        return img

    def gen():
        for p, m, l in zip(paths, meta, labels):
            yield (load_img(p), m), l

    ds = tf.data.Dataset.from_generator(
        gen,
        output_signature=(
            (tf.TensorSpec(shape=(224,224,3), dtype=tf.float32),
             tf.TensorSpec(shape=(META_DIM,), dtype=tf.float32)),
            tf.TensorSpec(shape=(NUM_CLASSES,), dtype=tf.float32)
        )
    )
    if shuffle:
        ds = ds.shuffle(1000, seed=42)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_tf_dataset(train_df, augment=True, shuffle=True)
val_ds = make_tf_dataset(val_df, augment=False, shuffle=False)
test_ds = make_tf_dataset(test_df, augment=False, shuffle=False)
print("Data pipeline ready.")

## STEP 6: Build ConvNeXtLarge + Metadata Fusion Model

In [ ]:
image_input = tf.keras.Input(shape=(224,224,3), name='image_input')
base_cnn = ConvNeXtLarge(weights='imagenet', include_top=False, input_shape=(224,224,3))
base_cnn.trainable = False
x_img = base_cnn(image_input, training=False)
x_img = layers.GlobalAveragePooling2D()(x_img)
x_img = layers.Dense(256, activation='relu')(x_img)
x_img = layers.Dropout(0.3)(x_img)

meta_input = tf.keras.Input(shape=(META_DIM,), name='meta_input')
x_meta = layers.Dense(32, activation='relu')(meta_input)
x_meta = layers.Dropout(0.2)(x_meta)
x_meta = layers.Dense(16, activation='relu')(x_meta)

fused = layers.Concatenate()([x_img, x_meta])
x = layers.Dense(128, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.001))(fused)
x = layers.Dropout(0.3)(x)
output = layers.Dense(NUM_CLASSES, activation='softmax')(x)

fusion_model = models.Model(inputs=[image_input, meta_input], outputs=output)
print("Total params:", fusion_model.count_params())

## STEP 7: PHASE 1 - Train Head Only (Frozen Base)
Expected: ~40-50 minutes. Model auto-saves to Drive after this phase completes.

In [ ]:
fusion_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
                      loss='categorical_crossentropy', metrics=['accuracy'])

es1 = callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
rlr1 = callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-7)

fusion_model.fit(train_ds, validation_data=val_ds, epochs=15,
                  callbacks=[es1, rlr1], class_weight=class_weights, verbose=1)

# Save immediately after Phase 1, in case Phase 2 gets interrupted
fusion_model.save(f'{SAVE_DIR}/convnextlarge_fusion_phase1.keras')
print("Phase 1 model saved to Drive!")

## STEP 8: PHASE 2 - Fine-Tune ConvNeXtLarge (Unfreeze Last Layers)
Expected: ~50-70 minutes.

In [ ]:
base_cnn.trainable = True
for layer in base_cnn.layers[:-20]:
    layer.trainable = False

fusion_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
                      loss='categorical_crossentropy', metrics=['accuracy'])

es2 = callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
rlr2 = callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-8)

fusion_model.fit(train_ds, validation_data=val_ds, epochs=15,
                  callbacks=[es2, rlr2], class_weight=class_weights, verbose=1)

fusion_model.save(f'{SAVE_DIR}/convnextlarge_fusion_final.keras')
print("Final fine-tuned model saved to Drive!")

## IF SESSION DISCONNECTED: Reload the Saved Model Instead of Retraining
Only run this if you had to restart your session after Phase 1 or Phase 2 already completed.

In [ ]:
# Uncomment ONE of these if you need to reload after a disconnect:
# fusion_model = tf.keras.models.load_model(f'{SAVE_DIR}/convnextlarge_fusion_final.keras')  # if Phase 2 finished
# fusion_model = tf.keras.models.load_model(f'{SAVE_DIR}/convnextlarge_fusion_phase1.keras')  # if only Phase 1 finished
# test_df = pd.read_csv(f'{SAVE_DIR}/test_df.csv')
print("Uncomment the relevant line above only if reloading after a disconnect.")

## STEP 9: Evaluate on Test Set (Standard, No TTA)

In [ ]:
y_pred_probs = fusion_model.predict(test_ds, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = test_df['label_idx'].values

acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
y_true_bin = label_binarize(y_true, classes=list(range(len(classes))))
auc = roc_auc_score(y_true_bin, y_pred_probs, average='macro', multi_class='ovr')

print(f"Standard Accuracy:  {acc:.4f}")
print(f"Standard F1-Score:  {f1:.4f}")
print(f"Standard AUC:       {auc:.4f}")
print(f"Vs base paper's best (87.62%): {(acc-0.8762)*100:+.2f} points")

## STEP 10: Apply Test-Time Augmentation (TTA) for Extra Boost
Predicts on 5 augmented versions of each test image and averages results. Takes a few minutes.

In [ ]:
def tta_predict(model, df, n_augments=5):
    all_preds = []
    for i in range(n_augments):
        augment = (i > 0)
        ds = make_tf_dataset(df, augment=augment, shuffle=False)
        preds = model.predict(ds, verbose=0)
        all_preds.append(preds)
    return np.mean(all_preds, axis=0)

print("Running TTA...")
tta_probs = tta_predict(fusion_model, test_df, n_augments=5)
tta_pred = np.argmax(tta_probs, axis=1)

tta_acc = accuracy_score(y_true, tta_pred)
tta_f1 = f1_score(y_true, tta_pred, average='macro', zero_division=0)
tta_auc = roc_auc_score(y_true_bin, tta_probs, average='macro', multi_class='ovr')

print("="*55)
print("FINAL RESULT WITH TTA:")
print("="*55)
print(f"Accuracy:  {tta_acc:.4f} (was {acc:.4f} without TTA)")
print(f"F1-Score:  {tta_f1:.4f} (was {f1:.4f} without TTA)")
print(f"AUC:       {tta_auc:.4f} (was {auc:.4f} without TTA)")
print(f"\nVs base paper's best (87.62%): {(tta_acc-0.8762)*100:+.2f} points")

np.save(f'{SAVE_DIR}/final_tta_probs.npy', tta_probs)

## STEP 11: Confusion Matrix + Per-Class Report (Final Model, with TTA)

In [ ]:
cm = confusion_matrix(y_true, tta_pred)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', xticklabels=classes, yticklabels=classes)
plt.title('Confusion Matrix - Final Model (ConvNeXtLarge Fusion + TTA)')
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/FINAL_confusion_matrix.png', dpi=300)
plt.show()
print(classification_report(y_true, tta_pred, target_names=classes))

## STEP 12: Complete Comparison Table (All Models So Far)

In [ ]:
final_table = pd.DataFrame({
    'Base Paper Best (ConvNeXtXLarge)': {'Accuracy': 0.8762, 'F1-Score': 0.7615, 'AUC': np.nan},
    'ResNet50 (my baseline, frozen, image-only)': {'Accuracy': 0.7824, 'F1-Score': 0.5427, 'AUC': 0.9365},
    'ConvNeXtBase Fusion (previous, single model)': {'Accuracy': 0.8024, 'F1-Score': 0.6937, 'AUC': 0.9614},
    'FINAL: ConvNeXtLarge Fusion + Fine-Tune + Class-Weight (no TTA)': {'Accuracy': acc, 'F1-Score': f1, 'AUC': auc},
    'FINAL: ConvNeXtLarge Fusion + Fine-Tune + Class-Weight + TTA': {'Accuracy': tta_acc, 'F1-Score': tta_f1, 'AUC': tta_auc},
}).T
print(final_table.round(4))
final_table.to_csv(f'{SAVE_DIR}/FINAL_complete_comparison.csv')

## STEP 13: Download Final Results

In [ ]:
from google.colab import files
files.download(f'{SAVE_DIR}/FINAL_complete_comparison.csv')
files.download(f'{SAVE_DIR}/FINAL_confusion_matrix.png')

---
## Summary for Your IEEE Paper

**Model:** ConvNeXtLarge backbone (same family as the base paper's best model) + patient metadata fusion (age, sex, lesion location) + fine-tuning + class-weighted loss + test-time augmentation.

**Novelty claim:** Unlike the base paper, which relies solely on frozen, image-only CNN features, this work integrates patient demographic and anatomical metadata with fine-tuned visual features to mirror real clinical decision-making, while explicitly addressing the severe class imbalance present in HAM10000 through class-weighted training.

Report both the overall accuracy/F1/AUC AND the per-class recall for melanoma and dermatofibroma -- these are your strongest, most defensible results regardless of the final overall accuracy number.